# **Sequence to Sequence RNN Models: Language Translation**

## Objectives


 - Comprehend recurrent neural networks (RNN) architecture
 - Create an Encoder-Decoder model for a translation task
 - Train and evaluate the model
 - Create a generator for the translation task
 - Explain concepts related to Perplexity and BLEU score and use them for evaluating translations


# Background

Sequence-to-sequence (Seq2seq) models have transformed the landscape of natural language processing (NLP), enabling breakthroughs in tasks such as speech recognition, dialogue systems, and code generation. These models leverage Recurrent Neural Networks (RNNs) to map variable-length input sequences to variable-length output sequences through an encoder-decoder architecture.

## History of Sequence-to-Sequence Models

Sequence-to-sequence models emerged from the limitations of fixed-size input/output neural networks.
Researchers identified the necessity for architectures capable of processing arbitrary-length sequences, particularly for language translation tasks.
The landmark contribution of Cho et al. (2014) established the encoder-decoder framework that forms the foundation of modern seq2seq models.

Here are some main applications of seq2seq models:
- **Speech Recognition**: Converting spoken audio sequences into written text sequences.
- **Code Generation**: Producing executable code from natural language descriptions.
- **Dialogue Systems**: Generating contextually appropriate responses in conversational AI.

And many more tasks that require mapping one sequence domain to another.

## Introduction to RNNs

RNNs are a specialized class of neural networks built to handle time-dependent and sequential data.
They maintain a hidden state ($h_t$) that acts as a compressed memory of all previously seen inputs.
RNNs utilize feedback connections that propagate information across time steps during processing.

Recurrent Neural Networks (RNNs) operate on sequences and utilize previous states to influence the current state. Here's the general formulation of a simple RNN:

**Given:**
- $\mathbf{x}_t$: input vector at time step $t$
- $\mathbf{h}_{t-1}$: hidden state vector from the previous time step
- $\mathbf{W}_x$ and $\mathbf{W}_h$: weight matrices for the input and hidden state, respectively
- $\mathbf{b}$: bias vector
- $\sigma$: activation function (often a sigmoid or tanh)

The update equation for the hidden state $\mathbf{h}_t$ is as follows:

$$
\mathbf{h}_t = \sigma(\mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + \mathbf{b})
$$

It can be observed that the hidden state at time $t$ is jointly determined by the current input and the previous hidden state, enabling the network to accumulate contextual information over time.

For the output (if you're making a prediction at each time step):

$$
\mathbf{y}_t = \text{softmax}(\mathbf{W}_o \cdot \mathbf{h}_t + \mathbf{b}_o)
$$

**Where:**
- $\mathbf{W}_o$: weight matrix for the output
- $\mathbf{b}_o$: bias vector for the output

Depending on the specific task, an RNN cell can either emit an output from $h_t$ or pass it forward exclusively as internal memory. To better understand how this memory mechanism operates in practice, consider the following state transition diagram:

![State Transition Diagram](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Screenshot%202023-10-19%20at%2011.29.23%E2%80%AFAM.png)

The diagram illustrates a state transition system with three distinct states, represented by the prominent purple circles. Each state is uniquely identified by a value of $h$: $h = -1$, $h = 0$, and $h = 1$.

### State $h = -1$
- Remains in the same state when $x = 1$ (shown by the yellow self-loop).
- Transitions to the $h = 0$ state upon receiving input $x = -1$ (indicated by the red arrow).

### State $h = 0$
- Shifts to the $h = -1$ state when input $x = 1$ is received (indicated by the red arrow).
- Advances to the $h = 1$ state when input $x = -1$ is received (indicated by the red arrow).

### State $h = 1$
- Remains in the same state when $x = -1$ (shown by the yellow self-loop).
- Transitions back to the $h = 0$ state upon receiving input $x = 1$ (indicated by the red arrow).

---

In summary, this diagram effectively captures the dynamic behavior of an RNN's hidden state, demonstrating how the network transitions between memory states based on sequential inputs $x$. Depending on both the current state and the incoming input, the system either shifts to a new state or holds its current configuration — mirroring how an RNN retains and updates its internal memory over time.


# LSTM and Sequence-to-Sequence Architecture

## Long Short-Term Memory (LSTM)

In real-world applications, standard RNNs are rarely used in their basic form. Instead, advanced variants such as **Long Short-Term Memory (LSTM)** and **Gated Recurrent Units (GRU)** are preferred, as they effectively overcome the vanishing gradient problem that limits basic RNNs.

An LSTM cell is built around three core mechanisms known as **gates**:

- The **Input Gate** regulates the flow of new information into the cell's memory. By examining both the current input and the prior hidden state, it selectively determines which portions of the incoming data are worth retaining.

- The **Forget Gate** is responsible for clearing out outdated or irrelevant information from memory. It evaluates the current input alongside the previous hidden state to decide which stored information is no longer useful and should be discarded.

- The **Output Gate** governs what portion of the cell's memory is passed forward as output. Based on the current input and previous hidden state, it filters the memory to produce a meaningful output signal.

The fundamental strength of LSTMs lies in their dedicated memory state, which can independently retain or release information as needed. This design enables LSTMs to effectively capture long-range dependencies and preserve critical context from earlier positions in a sequence.

---

## Sequence-to-Sequence Architecture

Seq2seq models are structured around an **Encoder-Decoder** framework. The encoder processes the entire input sequence and compresses it into a compact representation known as the **context vector** ($h_t$). The decoder then takes this context vector and uses it to generate the output sequence step by step.

### How It Works — Translation Example

Consider translating the English sentence *"I love to travel"* into French *"J'adore voyager"* — a classic seq2seq task.

**Encoder Phase:**
- Words from the input sentence are fed into the encoder one at a time.
- At each step, an RNN cell takes the current word ($x_t$) and its internal memory ($h_t$), processes them together, and passes an updated context vector ($h_{t+1}$) to the next cell.
- Once the entire input sequence is consumed, the final context vector captures a compressed understanding of the whole sentence.

**Decoder Phase:**
- The context vector is handed off to the decoder.
- The decoder consists of RNN cells that generate output words one at a time.
- At each step, a decoder cell receives the previously generated word along with the updated context vector from the prior cell, and produces the next output word ($y_t$).

This encoder-decoder design allows seq2seq models to generate output sequences of **arbitrary length**, making them highly flexible for tasks like translation, summarization, and dialogue generation.


# Encoder Implementation in PyTorch

## Overview

To build the encoder in PyTorch, you subclass `torch.nn.Module` and implement two essential methods: `__init__()` for defining the architecture and `forward()` for specifying the data flow.

---

## Constructor Parameters — `__init__()`

The following parameters configure the encoder's structure:

- **`vocab_len`** — Represents the total number of unique tokens in the vocabulary. This value is derived after preprocessing the dataset and serves as the dimensionality of the model's input.

- **`embedding_dim`** — Defines the size of the output embedding vector. For a small-scale demo application, values in the range of **256 to 512** are generally recommended.

- **`n_layers`** — Specifies how many LSTM layers to stack. While a single layer is used in the initial setup, this parameter is included upfront to allow easy scaling in future iterations.

- **`hid_dim`** — Controls the dimensionality of both the hidden state and the cell state within the LSTM.

- **`dropout`** — A regularization parameter that randomly deactivates a fraction of neurons during training to help prevent overfitting.

---

## Layer Definitions

Two core layers are defined inside `__init__()`:

- **Embedding Layer** — Maps raw input tokens to dense vector representations. Its input dimension is set to `vocab_len` and its output dimension to `embedding_dim`.

- **LSTM Layer** — Accepts the embedding vectors (of size `embedding_dim`) as input and produces three outputs: `output`, `hidden`, and `cell`. The internal capacity of the LSTM is controlled by `hid_dim`.

---

## Forward Pass — `forward()`

The data flows through the encoder as follows:

1. The **Embedding Layer** receives the input batch and internally maps each token to a one-hot representation before converting it into a dense embedding vector.
2. The **LSTM Layer** then processes these embeddings and produces three vectors: `output`, `hidden`, and `cell`.
3. Since the encoder's sole responsibility is to compress the input into a **context vector**, the `output` is discarded. Only `hidden` and `cell` are returned and passed on to the decoder.

---

> **Note:** When using an LSTM, the context vector consists of both the **hidden state** and the **cell state**. In contrast, if a GRU were used instead, only the **hidden state** would be available, as GRUs do not maintain a separate cell state.


### Import Libraries

In [31]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import random
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### **Define Encoder Class**

In [3]:
class Encoder(nn.Module):
    
    def __init__(self, vocab_len, embed_dim, hid_dim, n_layers, dropout_prob ):
        super().__init__()
        
        # define class attributes
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embed_dim = embed_dim
        self.vocab_len = vocab_len
        
        #define embedding layer for inputs
        self.embedding = nn.Embedding(self.vocab_len, self.embed_dim)
        #define LSTM layer
        self.lstm = nn.LSTM(embed_dim,hid_dim,n_layers,dropout = dropout_prob)
        # define drop out for randomly drop neurons
        self.dropout = nn.Dropout(dropout_prob)
        
    def forward(self, input_batch):
        #input_batch = [src len, batch size]
        embed_out = self.dropout(self.embedding(input_batch))
        embed_out = embed_out.to(device)
        
        #outputs = [src len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]
        
        output, (hidden,cell) = self.lstm(embed_out)
        
        return hidden, cell
        

Test the Encoder Class

In [4]:
vocab_size = 10
hid_dim  = 8
embded_dim = 10
n_layers = 1
dropout_prob = 0.5

encoder_t = Encoder(vocab_size,embded_dim,hid_dim,n_layers,dropout_prob)

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [5]:
src_data = torch.tensor([[0,3,4,2,1]])
src_data.shape

torch.Size([1, 5])

In [6]:
# so LSTM must be feed src len firstly. so we need to transpose the input   
src_data_t = src_data.t().to(device)
src_data_t.shape, src_data_t

(torch.Size([5, 1]),
 tensor([[0],
         [3],
         [4],
         [2],
         [1]]))

In [7]:
hidden_t, cell_t = encoder_t(src_data_t)
print("Hidden tensor from encoder:",hidden_t ,"\nCell tensor from encoder:", cell_t)

Hidden tensor from encoder: tensor([[[-0.3375,  0.1902,  0.0332,  0.0984,  0.2199,  0.0853, -0.0847,
           0.1432]]], grad_fn=<StackBackward0>) 
Cell tensor from encoder: tensor([[[-0.4240,  0.3697,  0.2506,  0.1965,  0.5036,  0.1501, -0.1367,
           0.4388]]], grad_fn=<StackBackward0>)


The encoder receives the entire source sequence as input, which is a sequence of words or tokens. This sequence is first converted into embeddings and then processed step-by-step by the LSTM. At each time step, the LSTM updates its hidden and cell states, which together store information about both short-term and long-term dependencies in the sequence.

As the LSTM processes the input, its hidden states act as a form of memory that captures the contextual meaning of the sequence over time. After the entire sequence has been processed, the final hidden state (along with the final cell state) represents a compressed summary of the input. This summary is often referred to as the context vector, as it encodes the overall meaning of the source sequence and is passed to the decoder.

### **Define Decoder Class**

The decoder class inherits from nn.Module, which is a base class for all neural network modules in PyTorch.
The constructor (__init__ method) initializes the parameters and layers of the decoder.
- `output_dim` is the number of possible output values(target vocab length).
- `emb_dim` is the dimensionality of the embedding layer.
- `hid_dim` is the dimensionality of the hidden state in the LSTM.
- `n_layers` is the number of layers in the LSTM.
- `dropout` is the dropout probability.

The decoder contains the following layers:
- `embedding`: An embedding layer that maps the output values to dense vectors of size emb_dim.
- `lstm`: An LSTM layer that takes the embedded input and produces hidden states of size hid_dim.
-  `fc_out`: A linear layer that maps the LSTM output to the output dimension output_dim.
- `softmax`: A log-softmax activation function applied to the output to obtain a probability distribution over the output values.
- `dropout`: A dropout layer that applies dropout to the embedded input.


In [8]:
class Decoder (nn.Module):
    
    def __init__(self, output_dim, embed_dim, hid_dim, n_layers, dropout):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.output_dim = output_dim
        
        #define embedded layer
        self.embedding = nn.Embedding(output_dim,embed_dim)
        #define lstm layer
        self.lstm = nn.LSTM(embed_dim,hid_dim,n_layers,dropout = dropout)
        # define output as fully connected network
        self.fc_out = nn.Linear(in_features=hid_dim, out_features=output_dim)
        #define softmax layer
        self.softmax = nn.LogSoftmax(dim=1)
        #define dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        
        #input = [batch size]

        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #n directions in the decoder will both always be 1, therefore:
        
        #hidden = [n layers, batch size, hid dim]
        #context = [n layers, batch size, hid dim]
        input = input.unsqueeze(0)
        #input = [1, batch size]
        
        embedded = self.dropout(self.embedding(input))
        #embedded = [1, batch size, emb dim]
        
        output, (hidden_out,cell_out)  = self.lstm(embedded,(hidden,cell))
        #output = [seq len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]
        
        #seq len and n directions will always be 1 in the decoder, therefore:
        #output = [1, batch size, hid dim]
        #hidden = [n layers, batch size, hid dim]
        #cell = [n layers, batch size, hid dim]
        
        prediction = self.fc_out(output.squeeze(0))
        
        prediction_prob = self.softmax(prediction)
        #prediction_prob = [batch size, output dim]
        
        return prediction_prob, hidden_out, cell_out

Test the Decoder

In [9]:
output_dim = 6
embed_dim = 10
hid_dim = 8
n_layers = 1
dropout_prob=0.5

decoder_t = Decoder(output_dim,embed_dim,hid_dim,n_layers,dropout_prob)

In [10]:
input_t = torch.tensor([0]).to(device) #<bos>
input_t.shape

torch.Size([1])

In [11]:
prediction_prob, hidden_decoder, cell_decorder = decoder_t(input_t,hidden_t,cell_t)
print("Prediction:", prediction_prob, '\nHidden:',hidden_decoder,'\nCell:', cell_decorder)

Prediction: tensor([[-1.7035, -1.5524, -1.9980, -1.7770, -1.8652, -1.9201]],
       grad_fn=<LogSoftmaxBackward0>) 
Hidden: tensor([[[-0.4917,  0.0961,  0.0858, -0.1614,  0.3907, -0.0303, -0.0461,
          -0.1196]]], grad_fn=<StackBackward0>) 
Cell: tensor([[[-0.5768,  0.3313,  0.1018, -0.3952,  0.5139, -0.0464, -0.1037,
          -0.1997]]], grad_fn=<StackBackward0>)


### **Encoder - Decorder Connection**

teacher_forcing_ratio = probability of using ground truth token

In [12]:
# target = [target_len, batch_size]
teacher_force_ratio = 0.5
target = torch.tensor([[0],[2],[3],[5],[1]]).to(device)

target.shape

torch.Size([5, 1])

In [13]:
batch_size = target.shape[1]
target_len =  target.shape[0]
batch_size, target_len

(1, 5)

In [14]:
target_vocab_size = decoder_t.output_dim
target_vocab_size

6

create a tensor to store the decorder output

In [15]:
output_t =  torch.zeros(target_len,batch_size,target_vocab_size).to(device)
output_t, output_t.shape

(tensor([[[0., 0., 0., 0., 0., 0.]],
 
         [[0., 0., 0., 0., 0., 0.]],
 
         [[0., 0., 0., 0., 0., 0.]],
 
         [[0., 0., 0., 0., 0., 0.]],
 
         [[0., 0., 0., 0., 0., 0.]]]),
 torch.Size([5, 1, 6]))

In [16]:
hidden_t = hidden_t.to(device)
cell_t = cell_t.to(device)

In [17]:
#first input to the decoder is the <bos> tokens
input = target[0,:]

for t in range(1, target_len):
    
    #you loop through the trg len and generate tokens
    #decoder receives previous generated token, cell and hidden
    #decoder outputs it prediction(probablity distribution for the next token) and updates hidden and cell
    output_t_update, hidden_t_update, cell_t_update = decoder_t(input, hidden_t, cell_t)
    #replace output value of the defined zero output tensor
    output_t[t] = output_t_update
    
    #decide if you are going to use teacher forcing or not
    teacher_force = random.random() < teacher_force_ratio
    
    #get the highest predicted token from your predictions
    prediction = output_t_update.argmax(1)
    
    #if teacher forcing, use actual next token as next input
    #if not, use predicted token
    #input = trg[t] if teacher_force else top1
    
    input = target[t] if teacher_force else prediction

print(output_t,output_t.shape )

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],

        [[-1.8054, -1.4835, -2.0132, -1.8578, -1.8072, -1.8639]],

        [[-1.6176, -1.6772, -2.0582, -1.8693, -1.6959, -1.9012]],

        [[-1.6841, -1.6487, -2.0672, -1.8048, -1.7202, -1.8837]],

        [[-1.5259, -1.6832, -2.0025, -1.8688, -1.7437, -2.0202]]],
       grad_fn=<CopySlices>) torch.Size([5, 1, 6])


The size of output tensor is (trg_len, batch_size, trg_vocab_size). This is because for each `trg` token (length of `trg`) the model outputs a probability distribution over all possible tokens(trg vocab length). Therefore, to generate the predicted tokens or translation of the `src` sentence, you need to get the maximum probability for each token:


In [18]:
pred_tokens = output_t.argmax(2)
print(pred_tokens, pred_tokens.shape)

tensor([[0],
        [1],
        [0],
        [1],
        [0]]) torch.Size([5, 1])


The output is not corrected because we didn't combined all together and train the model

Let's put together all the code for connecting the encoder and decoder in a seq2seq class for better usability.


## **Sequence to Sequence Model**

## Sequence-to-sequence model implementation in PyTorch  
Now, let’s combine the encoder and decoder to build a complete seq2seq architecture.

You create a `seq2seq` class by extending `nn.Module`, which is the fundamental class for all neural network models in PyTorch.  
The inputs to this class include:  
- `encoder` and `decoder`, which are the previously defined network components.  
- `device`, indicating whether computations run on CPU or GPU.  
- `trg_vocab`, representing the target language vocabulary, mainly used to define the output dimension.

The **forward** method specifies how data flows through the model. It accepts three inputs: `src`, `trg`, and `teacher_forcing_ratio`.

- `src` refers to the input (source) sequence, while `trg` is the expected output (target) sequence.  
- `teacher_forcing_ratio` controls the likelihood of applying teacher forcing during training. With teacher forcing, the actual target token is fed into the decoder instead of the model’s own previous prediction.

Inside the **forward** method, key variables such as `batch_size`, `trg_len`, and `trg_vocab_size` are initialized. An empty tensor named `outputs` is also created to collect predictions at each time step.

The encoder processes the source sequence to generate `hidden` and `cell` states. These states act as the starting point for the decoder.

For the first decoding step, the input is the `<bos>` (beginning-of-sequence) token from the target sequence.

The decoder then runs step-by-step through the target sequence (`for t in range(1, trg_len)`). At each step, it takes the current input along with the previous hidden and cell states to produce an output. This output is saved into the `outputs` tensor.

During each time step, the model decides whether to use teacher forcing. If selected, the actual next token from `trg[t]` is used as input. Otherwise, the model uses its own predicted token (`top1 = output.argmax(1)`).

In the end, the method returns the `outputs` tensor, which contains predictions for the entire sequence.

### Define Seq Class

In [72]:
class SeqToSeq(nn.Module):
    
    def __init__(self, encoder:Encoder, decoder:Decoder,device, target_vocab):
        super().__init__()
        
        self.encoder = encoder
        self.decorder = decoder
        self.device = device
        self.target_vocab = target_vocab
        
        # assert is a debugging statement in Python that checks a condition:

        #     If the condition is True → nothing happens, the program continues.
        #     If the condition is False → the program stops immediately and raises an error with the given message.
        
        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal"
        assert encoder.n_layers == decoder.n_layers, \
            "n_layers of encoder and decoder must be equal"
        
    def forward(self, src, target, teacher_force_ratio):
        # src = [src_len, batch_size]
        # target = [target_len, batch_size]
        
        batch_size = target.shape[1]
        target_len = target.shape[0]
        
        target_vocab_size = self.decorder.output_dim
        
        # create a zero tensor to store the decorder outputs
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(self.device)
        
        hidden, cell = self.encoder(src)
        hidden = hidden.to(device)
        cell = cell.to(device)
        
        # define decorder first input
        input = target[0,:]
        
        
        for t in range(1, target_len):
            #insert input token embedding, previous hidden and previous cell states
            #receive output tensor (predictions) and new hidden and cell states
            output_de, hidden_de, cell_de = self.decorder(input, hidden, cell)
            outputs[t] = output_de
            
            teacher_force = random.random() < teacher_force_ratio
            
            output_value = output_de.argmax(1)
            
            # set input for next token base on teacher ratio
            input = target[t] if teacher_force else output_value
        
        return outputs

In [73]:
from tqdm import tqdm

In [225]:
def train(model, iterator, optimizer, criterion, clip, num_epochs=10, teacher_forcing_ratio=0.5):
    
    average_epoch_loss = []
    
    for epoch in range(num_epochs):
        
        model.train()
        epoch_loss = 0
        
        train_iterator = tqdm(iterator, desc=f"Epoch {epoch+1}", leave=False)
        
        for i, (src, target) in enumerate(train_iterator):
            
            src = src.to(device)
            target = target.to(device)
            
            optimizer.zero_grad()
            
            output = model(src, target, teacher_forcing_ratio)
            
            output_dim = output.shape[-1]
            
            output = output[1:].view(-1, output_dim)
            target = target[1:].contiguous().view(-1)
            
            loss = criterion(output, target)
            
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            
            optimizer.step()
            
            train_iterator.set_postfix(loss=loss.item())
            
            epoch_loss += loss.item()
        
        average_epoch_loss.append(epoch_loss / len(iterator))
    
    return average_epoch_loss

## Evaluating model in PyTorch
You also need to define a function to evaluate the model. Let's go through the code and understand its components:

1. `evaluate(model, iterator, criterion)` takes three arguments:
   - `model` is the neural network model that will be evaluated.
   - `iterator` is an iterable object that provides the evaluation data in batches.
   - `criterion` is the loss function that measures the model's performance.
* Note that evaluate function do not perform any optimization on the model.

2. The function starts by setting the model to evaluation mode with `model.eval()`.

3. It initializes a variable `epoch_loss` to keep track of the accumulated loss during the evaluation.

4. The function enters a `with torch.no_grad()` block, which ensures that no gradients are computed during the evaluation. This saves memory and speeds up the evaluation process since gradients are not needed for parameter updates.

5. The function iterates over the evaluation data provided by the `iterator`. Each iteration retrieves a batch of input sequences (`src`) and target sequences (`trg`).

6. The input sequences (`src`) and target sequences (`trg`) are moved to the appropriate device (e.g., GPU) using `src = src.to(device)` and `trg = trg.to(device)`.

7. The model is then called with `output = model(src, trg, 0)` to obtain the model's predictions for the target sequences. The third argument `0` is passed to indicate that teacher forcing is turned off during evaluation.  During evaluation, teacher forcing is typically turned off to evaluate the model's ability to generate sequences based on its own predictions.

8. The `output` tensor has dimensions `[trg len, batch size, output dim]`. To calculate the loss, the tensor is reshaped to `[trg len - 1, batch size, output dim]` to remove the initial `<bos>` (beginning of sequence) token, which is not used for calculating the loss.

9. The target sequences (`trg`) are also reshaped to `[trg len - 1]` by removing the initial `<bos>` token and making it a contiguous tensor. This matches the shape of the reshaped `output` tensor.

10. The loss between the reshaped `output` and `trg` tensors is calculated using the specified `criterion`.

11. The current batch loss (`loss.item()`) is added to the `epoch_loss` variable.

12. After all the batches have been processed, the function returns the average loss per batch for the entire evaluation, calculated as `epoch_loss / len(list(iterator))`.


### **Define Evaluation Function**

In [226]:
def model_eval(iterator, model,critertion ):
    
    model.eval()
    
    epoch_loss = 0
    
    valid_loader = tqdm(iterator, desc="Validating", leave=False)
    
    with torch.no_grad():
        
        for i, (src, target) in enumerate(valid_loader):
            # convert data laoder into device
            src = src.to(device)
            target = target.to(device)
            # predict model output
            output = model(src,target,0) # teacher_forcing_ratio = 0
            
            #trg = [trg len, batch size]
            #output = [trg len, batch size, output dim]
            output_dim = output.shape[-1]
            
            #flattern the output
            output = output[1:].view(-1,output_dim)
            target = target[-1:].contiguous.view(-1)
            #trg = [(trg len - 1) * batch size]
            #output = [(trg len - 1) * batch size, output dim]
            
            loss = critertion(output, target)
            
            epoch_loss += loss.item()
            
    return epoch_loss/len(list(iterator))
            

## **Data Pre-Processing**

In this section, you will fetch a language translation dataset called Multi30k, collate it (tokenization, numericalization, and adding BOS/EOS and padding) and create iterable batches of src and trg tensors.

This leverages the predefined collate_fn to efficiently curate and ready batches for training the transformer model. The primary aim is to delve deeper into the intricacies of the RNN encoder and decoder components.


In [227]:
import requests

In [228]:
def download(url, fileName):
    response = requests.get(url)
    if response.status_code == 200:
        with open(fileName,"wb") as f:
            f.write(response.content)
    

In [229]:
fileName = "Multi30K_de_en_dataloader.py"
url= 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Multi30K_de_en_dataloader.py'

In [230]:
download(url,fileName)

### Import Data Loaders

In [231]:
from Multi30K_de_en_dataloader import get_translation_dataloaders, index_to_eng,index_to_german

In [244]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torchtext.datasets import Multi30k, multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from typing import Iterable, List

# We need to modify the URLs for the dataset since the links to the original dataset are broken
multi30k.URL["train"] = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/training.tar.gz"
multi30k.URL["valid"] = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/validation.tar.gz"

SRC_LANGUAGE = 'de'
TGT_LANGUAGE = 'en'

# Making a placeholder dict to store both tokenizers
token_transform = {}
token_transform[SRC_LANGUAGE] = get_tokenizer('spacy', language='de_core_news_sm')
token_transform[TGT_LANGUAGE] = get_tokenizer('spacy', language='en_core_web_sm')

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

# Place holder dict for 'en' and 'de' vocab transforms
vocab_transform = {}

def yield_tokens(data_iter: Iterable, language: str) -> List[str]:
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}
    for data_sample in data_iter:
        yield token_transform[language](data_sample[language_index[language]])

for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    train_iterator = Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    sorted_dataset = sorted(train_iterator, key=lambda x: len(x[0].split()))
    vocab_transform[ln] = build_vocab_from_iterator(yield_tokens(sorted_dataset, ln),
                                                    min_freq=1,
                                                    specials=special_symbols,
                                                    special_first=True)

for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[ln].set_default_index(UNK_IDX)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def tensor_transform_s(token_ids: List[int]):
    return torch.cat((torch.tensor([BOS_IDX]),
                      torch.flip(torch.tensor(token_ids), dims=(0,)),
                      torch.tensor([EOS_IDX])))

def tensor_transform_t(token_ids: List[int]):
    return torch.cat((torch.tensor([BOS_IDX]),
                      torch.tensor(token_ids),
                      torch.tensor([EOS_IDX])))

def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

text_transform = {}
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_sequences = text_transform[SRC_LANGUAGE](src_sample.rstrip("\n"))
        src_sequences = torch.tensor(src_sequences, dtype=torch.int64)
        tgt_sequences = text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n"))
        tgt_sequences = torch.tensor(tgt_sequences, dtype=torch.int64)
        src_batch.append(src_sequences)
        tgt_batch.append(tgt_sequences)

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    src_batch = src_batch.t()
    tgt_batch = tgt_batch.t()
    return src_batch.to(device), tgt_batch.to(device)

def get_translation_dataloaders(batch_size=4,flip=False):
    train_iterator = Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    sorted_train_iterator = sorted(train_iterator, key=lambda x: len(x[0].split()))
    # Update text_transform based on the flip parameter
    if flip:
        text_transform[SRC_LANGUAGE] = sequential_transforms(token_transform[SRC_LANGUAGE], vocab_transform[SRC_LANGUAGE], tensor_transform_s)
    else:
        text_transform[SRC_LANGUAGE] = sequential_transforms(token_transform[SRC_LANGUAGE], vocab_transform[SRC_LANGUAGE], tensor_transform_t)
    text_transform[TGT_LANGUAGE] = sequential_transforms(token_transform[TGT_LANGUAGE], vocab_transform[TGT_LANGUAGE], tensor_transform_t)
    
    train_dataloader = DataLoader(sorted_train_iterator, batch_size=batch_size, collate_fn=collate_fn, drop_last=True)

    valid_iterator = Multi30k(split='valid', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    sorted_valid_dataloader = sorted(valid_iterator, key=lambda x: len(x[0].split()))
    valid_dataloader = DataLoader(sorted_valid_dataloader, batch_size=batch_size, collate_fn=collate_fn, drop_last=True)

    return train_dataloader, valid_dataloader

def index_to_eng(seq_en):
    return " ".join([vocab_transform['en'].get_itos()[index.item()] for index in seq_en])

def index_to_german(seq_de):
    return " ".join([vocab_transform['de'].get_itos()[index.item()] for index in seq_de])


In [245]:
train_dataloader, valid_dataloader = get_translation_dataloaders(batch_size=4, flip=True)

In [246]:
src, trg = next(iter(train_dataloader))
src, trg

C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:68: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  src_sequences = torch.tensor(src_sequences, dtype=torch.int64)
C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt_sequences = torch.tensor(tgt_sequences, dtype=torch.int64)


(tensor([[    2,     2,     2,     2],
         [    3,  5510,  5510,  1701],
         [    1,     3,     3,     8],
         [    1,     1,     1, 12642],
         [    1,     1,     1,     3]]),
 tensor([[   2,    2,    2,    2],
         [   3, 6650,  216,    6],
         [   1, 4623,  110, 3398],
         [   1,  259, 3913,  202],
         [   1,  172, 1650,  109],
         [   1, 9953, 3823,   37],
         [   1,  115,   71,    3],
         [   1,  692, 2808,    1],
         [   1, 3428, 2187,    1],
         [   1,    5,    5,    1],
         [   1,    3,    3,    1]]))

English to German and German to English

In [247]:
data_itr = iter(train_dataloader)

for n in range(1000):
    german, english = next(data_itr)
    
for n in range(3):
    german, english = next(data_itr)
    german = german.T
    english = english.T
    
    print("________________")
    print("german")
    
    for g in german:
        print(index_to_german(g))
        
    print("________________")
    print("english")
    
    for e in english:
        print(index_to_eng(e))

C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:68: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  src_sequences = torch.tensor(src_sequences, dtype=torch.int64)
C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt_sequences = torch.tensor(tgt_sequences, dtype=torch.int64)


________________
german
<bos> . Innenstadt der in Hüten schwarzen mit Personen <eos>
<bos> . Stadt einer in protestiert Menschen Gruppe Eine <eos>
<bos> . mit Ansichten politischen ihre teilt Gruppe Eine <eos>
<bos> . Strand felsigen einem an sitzen Personen Mehrere <eos>
________________
english
<bos> People in black hats gathered together downtown . <eos> <pad> <pad> <pad>
<bos> A group of people protesting in a city . <eos> <pad> <pad>
<bos> A group is letting their political opinion be known . <eos> <pad>
<bos> A group of people are sitting on a rocky beach . <eos>
________________
german
<bos> . Sonnenbrillen und Hüten mit Personen sitzende Zwei <eos>
<bos> . Angeln beim Hut mit Junge kleiner Ein <eos>
<bos> . Giorgio's im Spaß haben Frauen zwei Diese <eos>
<bos> . Sofa dem auf schlafen Kinder kleine Zwei <eos>
________________
english
<bos> Two people sitting in hats and shades . <eos> <pad> <pad> <pad>
<bos> A young boy in a hat is fishing by himself . <eos>
<bos> These two wome

# 🧠 Understanding Sequence Representation in PyTorch

------------------------------------------------------------------------

## 🧠 1. Normal Tensor Thinking (Tabular Mindset)

Usually in PyTorch, we think like this:

    [batch_size, features]

### Example:

    [
     [age, salary],
     [age, salary],
     [age, salary]
    ]

👉 Rows = samples\
👉 Columns = features

------------------------------------------------------------------------

## 🔄 2. Sequence Data Changes Everything

Now imagine **sentences instead of rows**:

    "I love AI"
    "Deep learning is fun"
    "Hello"

Each sentence has **different lengths** ❗

So we can't directly put them into a tensor.

------------------------------------------------------------------------

## 🧱 3. Padding is Needed

We make all sentences equal length by adding `<pad>` tokens:

    "I love AI <pad>"
    "Deep learning is fun"
    "Hello <pad> <pad> <pad>"

Now all have length = 4

------------------------------------------------------------------------

## 🔁 4. The Important Twist (PyTorch Convention)

Instead of storing as:

    [batch_size, sequence_length]

PyTorch RNNs expect:

    [sequence_length, batch_size]

------------------------------------------------------------------------

## 👀 5. Visual Representation (VERY IMPORTANT)

Let's say we have 3 sentences after padding:

    Sentence 1: I      love     AI      <pad>
    Sentence 2: Deep   learning is      fun
    Sentence 3: Hello  <pad>    <pad>   <pad>

Now PyTorch stores it like this:

    [
     [I,      Deep,     Hello],   ← timestep 1
     [love,   learning, <pad>],   ← timestep 2
     [AI,     is,       <pad>],   ← timestep 3
     [<pad>,  fun,      <pad>]    ← timestep 4
    ]

### Shape:

    [sequence_length = 4, batch_size = 3]

------------------------------------------------------------------------

## 💡 Key Insight

👉 Each **row = one time step**\
👉 Each **column = one sentence**

------------------------------------------------------------------------

## 🤖 6. Why This Format?

Because models like:

-   RNN\
-   LSTM\
-   GRU

process sequences **step by step over time**

------------------------------------------------------------------------

## 📦 7. Where `pad_sequence` Comes In

When you use:

``` python
pad_sequence(list_of_tensors)
```

It:

1.  Finds the longest sequence\
2.  Pads shorter ones\
3.  Stacks them into:

```{=html}
<!-- -->
```
    [sequence_length, batch_size]

------------------------------------------------------------------------

## ⚠️ Common Confusion (Important)

👉 Padding happens **within each sequence**\
👉 But after stacking, it *looks like* it affects rows

------------------------------------------------------------------------

## 🔄 8. Alternative Format (`batch_first=True`)

``` python
nn.LSTM(..., batch_first=True)
```

Shape:

    [batch_size, sequence_length, feature_size]

------------------------------------------------------------------------

## 🧩 9. Final Mental Model

  Dimension    Meaning
  ------------ ------------
  1st (rows)   Time steps
  2nd (cols)   Sentences

------------------------------------------------------------------------

## 🎯 One-Line Summary

👉 Instead of **"one row = one sentence"**,\
it becomes **"one row = one time step across all sentences"**


In [248]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True


### Training
Now, define an instance of the model:

- `enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)`: This line creates an instance of the `Encoder` class, which represents the encoder component of the Seq2Seq model. The `Encoder` class takes the input dimension, embedding dimension, hidden dimension, number of layers, and dropout probability as arguments.

- `dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)`: This line creates an instance of the `Decoder` class, which represents the decoder component of the Seq2Seq model. The `Decoder` class takes the output dimension, embedding dimension, hidden dimension, number of layers, and dropout probability as arguments.

- `model = Seq2Seq(enc, dec, device,trg_vocab = vocab_transform['en']).to(device)`: This line creates an instance of the `Seq2Seq` class, which represents the entire Seq2Seq model. The `Seq2Seq` class takes the encoder, decoder, and device (e.g., CPU or GPU) as arguments. It combines the encoder and decoder to form the complete Seq2Seq architecture.


### **Model Paramter Initializatiob**

In [249]:
from Multi30K_de_en_dataloader import vocab_transform

In [250]:
input_dim = len(vocab_transform['de'])
output_dim = len(vocab_transform['en'])

enc_embed_dim = 128
dec_embed_dim = 128

hid_dim = 256
n_layers = 1

enc_dropout = 0.3
dec_dropout = 0.3

In [251]:
encoder = Encoder(input_dim,enc_embed_dim,hid_dim,n_layers,enc_dropout)
decoder = Decoder(output_dim,dec_embed_dim,hid_dim,n_layers,dec_dropout)

In [252]:
seq_model = SeqToSeq(encoder, decoder,device, target_vocab=vocab_transform['en']).to(device)

`def init_weights(m)`defines a function named `init_weights` that takes a module `m` as input. The purpose of this function is to initialize the weights of the neural network module.

The next line `for name, param in m.named_parameters():` starts a loop that iterates over the named parameters of the module `m`. Each parameter is accessed as `param` and its corresponding name is accessed as `name`.

`nn.init.uniform_(param.data, -0.08, 0.08)`initializes the parameter's data with values drawn from a uniform distribution between `-0.08` and `0.08`. The `nn.init.uniform_` function is provided by the PyTorch library and is used to initialize the weights of neural network parameters.

Finally, `model.apply(init_weights)` applies the `init_weights` function to the `model` instance. This ensures that the weights of all the parameters in the model are initialized using the specified uniform distribution.


In [253]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

seq_model.apply(init_weights)

SeqToSeq(
  (encoder): Encoder(
    (embedding): Embedding(19214, 128)
    (lstm): LSTM(128, 256, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decorder): Decoder(
    (embedding): Embedding(10837, 128)
    (lstm): LSTM(128, 256, dropout=0.3)
    (fc_out): Linear(in_features=256, out_features=10837, bias=True)
    (softmax): LogSoftmax(dim=1)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (target_vocab): Vocab()
)

This code defines a function `count_parameters` that counts the number of trainable parameters in a given model. It then prints the count of trainable parameters in a formatted string.


In [254]:
def count_parameters (model):
    
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"The model has {count_parameters(seq_model):,} trainable parameters")

The model has 7,422,165 trainable parameters


# Optimizer and Loss Function Configuration

This section describes how the optimizer and loss function are defined for training the sequence-to-sequence model.

---

## 1. Adam Optimizer

optimizer = optim.Adam(model.parameters())

# Understanding PAD_IDX and Criterion in Model Training

This section explains where `PAD_IDX` and `criterion` come from and how they are used in training a sequence model.

## Padding Index (PAD_IDX)

python
PAD_IDX = vocab_transform['en'].get_stoi()['<pad>']

In [255]:
optimizer = torch.optim.Adam(seq_model.parameters())
PAD_IDX = vocab_transform['en'].get_stoi()['<pad>']

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)

### Model Training

In [257]:
average_loss = train(seq_model,train_dataloader,optimizer,criterion,clip=1,num_epochs=3)

Epoch 1:   0%|          | 0/7250 [00:00<?, ?it/s]C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:68: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  src_sequences = torch.tensor(src_sequences, dtype=torch.int64)
C:\Users\Vish\AppData\Local\Temp\ipykernel_17232\751496865.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt_sequences = torch.tensor(tgt_sequences, dtype=torch.int64)
